# Step 4 Assembly
Cuts every raw table of the three London PICU sites (RBH, SMH, GOSH) down to the stays in the manifest and writes one parquet file per table to `derived/assembled/<SITE>/`.

For each table: keep only rows that belong to a manifest stay, drop exact duplicate rows, write the result. No column is dropped, no value is changed, and every column is written as text exactly as in the source. Anonymisation is a later step.

Run order: 1 inventory, 2 diagnostics, 3 manifest, then this notebook. The assembled files hold row-level data, so they stay in `derived/` (git-ignored) inside the SDE. The logs and reports here are counts only.

Assembly takes about 10 minutes, so each site runs once and its log is saved next to a provenance file (`<SITE>_assembly_log.json`) recording the manifest fingerprint, a hash of the assembly code and the settings it was made from. A rerun loads the saved log and checks that provenance. If the manifests, the assembly code or `time_cols` have changed, it stops with "stale: rerun with `rebuild = True`" and nothing is loaded. After an intended change, set `rebuild = True`, run, and read section 5: the file counts are compared with `Assembled_counts_baseline.csv`, so any difference is shown table by table. Update the baseline only after reading and accepting what changed.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import inventory as I
import assemble as A
import diagnostics as D

In [ ]:
smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
gosh_dir = Path("/home/jormond/Documents/DS217/users/jormond/GOSH")
derived = open_picu_dir / "derived"
out_dir = derived / "assembled"
reports_dir = open_picu_dir / "reports"
out_dir.mkdir(exist_ok=True)
rebuild = False        # True rewrites every assembled parquet of the sites below (about 10 minutes)
time_cols = {          # GOSH tables whose time column is not observation_datetime or hour_start
    "July2026extraction_by_alex/pAIC/merged_antibiotics/antibiotics.csv": "chartTime",
    "merged_balance/Net_Bal.csv": "chartTime",
    "merged_balance/Net_Bal_24.csv": "chartTime",
    "merged_balance/Total_fluids.csv": "chartTime",
    "merged_vent_data/vent_Ventilation_mode_flowsheet_rows.csv": "chart_time",
}

rbh_m = pd.read_csv(derived / "RBH_manifest.csv", dtype=str)
smh_m = pd.read_csv(derived / "SMH_manifest.csv", dtype=str)
gosh_m = pd.read_csv(derived / "GOSH_manifest.csv", parse_dates=["adm_time", "disch_time"])

## 1. Which tables will be assembled
Lists the tables of each site (headers only, so it is quick) and says which can be linked to a stay. Tables that cannot be linked are skipped and listed.

Defaults: RBH skips the old PICANet file; SMH skips the two medication files that `medications_v2` already combines; GOSH uses only the merged per-variable files (not the raw `lab_components`, the archived PICANet file or the old parquet). The manifests decide which stays are kept:
- **RBH:** matched by `stay_key` (patient and admission time, NIC/NND prefix removed).
- **SMH:** matched by `encounterId`.
- **GOSH:** matched by `project_id` and time. A row belongs to the stay whose window, from admission start to discharge, contains its timestamp. The supplier's `admission_identifier` is not used.

In [ ]:
for site, d in [("RBH", rbh_dir), ("SMH", smh_dir), ("GOSH", gosh_dir)]:
    print(site)
    A.list_tables(site, d)

## 2. RBH
For each table: read in chunks, replace `studyID` by the stay key, keep rows of manifest stays, remove duplicates. Two counts are reported separately:
- **exact duplicates removed:** identical rows.
- **merged (prefix only):** rows that differ only in the NIC/NND prefix. The same stay can appear under both prefixes in different tables (about 2,500 stays in the sedation table), so these are copies of one record, not two measurements.

In [ ]:
rbh_log = A.assemble_or_load("RBH", rbh_dir, out_dir, rbh_m, reports_dir / "RBH_assembly_log.csv", rebuild)
rbh_summary = A.summarise_log(rbh_log)

## 3. SMH
Keeps rows whose `encounterId` (`encounterID` in the medication files) is in the manifest, then removes exact duplicate rows: rows identical in every column. That decision is made here. The long-format tables (vent, vitals, airway, blood gas, labs) lose 29 to 52% of their rows this way, which the inventory predicted. Whether those rows are repeated copies from the extraction or real readings with identical fields has not been established; it is an open question for the SMH colleagues (`SMH_findings.md`, question 5). The harmonise step also removes repeats at the hourly level for several concepts, so the choice should matter little for hourly values, but it could matter for summed amounts such as intake and output.

In [ ]:
smh_log = A.assemble_or_load("SMH", smh_dir, out_dir, smh_m, reports_dir / "SMH_assembly_log.csv", rebuild)
smh_summary = A.summarise_log(smh_log)

## 4. GOSH
GOSH has no stay key in its tables, so rows are attached to stays by patient and time: the patient comes from `project_id`, and the stay is the one for that patient whose window (admission start with no margin to discharge) contains the row's time. Rows outside every stay (other wards, theatres, readmission gaps) are dropped, and `stay_id` is added. The `eprweights` PICANet table is written with all its columns plus `stay_id`. Tables without `project_id` or a time column are not assembled and are listed at the end of the run.

Some tables have no standard time column (`observation_datetime`, `hour_start`). For those the time column is set in `time_cols` (cell 2), after checking it.

In [ ]:
pick = r"caboodle_fluid_balance|Total_fluids"
A.list_tables("GOSH", gosh_dir, include=pick)
A.time_candidates("GOSH", gosh_dir, include=pick)

### Time columns of the balance and flowsheet tables
`Net_Bal`, `Net_Bal_24` and the ventilation-mode flowsheet rows each have `chartTime` (or `chart_time`), `start_datetime` and `end_datetime`. If the start and end columns were an admission window repeated on every row, they would not be measurement times. This samples the first 500,000 rows of each table and prints counts only: how many rows parse, the date range, how many patients have one value in the column, and how often the columns agree to the minute.

In [ ]:
for rel in ["merged_balance/Net_Bal.csv", "merged_balance/Net_Bal_24.csv"]:
    A.time_column_check(gosh_dir, rel, ["chartTime", "start_datetime", "end_datetime"])
A.time_column_check(gosh_dir, "merged_vent_data/vent_Ventilation_mode_flowsheet_rows.csv",
                    ["chart_time", "start_datetime", "end_datetime"])

In [ ]:
gosh_log = A.assemble_or_load("GOSH", gosh_dir, out_dir, gosh_m, reports_dir / "GOSH_assembly_log.csv",
                              rebuild, time_cols=time_cols)
gosh_summary = A.summarise_log(gosh_log)

## 5. Check against the files on disk and the baseline
Three checks that need no rebuild. First, the rows in each assembled parquet (read from the file metadata only) are compared with the rows written according to the log, table by table. Second, they are compared with the counts recorded on 2026-11-27 (`Assembled_counts_baseline.csv`: RBH 16 tables and 7,399,421 rows, SMH 11 and 18,960,001, GOSH 67 and 45,713,105). Third, the logs' totals are listed per site. Every `same` column must be `True`. After a rebuild this is the proof that nothing changed.

In [ ]:
counts = A.parquet_counts(out_dir)
baseline = pd.read_csv(reports_dir / "Assembled_counts_baseline.csv")
vs_baseline = A.compare_counts(baseline, counts)
vs_log = pd.concat([A.log_vs_disk(log, counts, site) for site, log in
                    (("RBH", rbh_log), ("SMH", smh_log), ("GOSH", gosh_log))], keys=["RBH", "SMH", "GOSH"])
totals = counts.groupby("site").agg(tables=("table", "count"), rows=("rows", "sum"))
assert vs_baseline["same"].all() and vs_log["same"].all(), "assembled files differ from the log or the baseline"
D.show({"Totals on disk": totals, "Rows by table, now against baseline": vs_baseline.groupby("site")["same"].agg(["count", "all"])})

## 6. One row per stay? (demographics, outcomes, PICANet)
These tables should have one row per stay, but a stay can have several rows that are not exact duplicates (repeated demographics rows, conflicting outcomes). This counts rows per stay for each table and lists the columns that differ between a stay's rows. If a stay's rows disagree on outcome or discharge status, we need a rule for choosing before anonymisation. The static step (step 5) excludes the few stays with two conflicting PICANet rows (1 RBH, 2 SMH).

In [ ]:
for site, key, names in [
    ("RBH", "stay_key", ["ai_picu_demographics", "ai_picu_outcomes", "ai_picu_piccanet"]),
    ("SMH", "encounterId", ["ai_picu_demographics", "ai_picu_outcomes", "ai_picu_piccanet"]),
]:
    for n in names:
        A.per_stay_conflicts(out_dir / site / f"{n}.parquet", key)
        print()

## 7. Save
Writes the counts report for this step: the logs' totals, the table counts on disk and the checks above. The three assembly logs and their provenance files were already saved by the cells that ran each site. Counts only.

In [ ]:
D.write_report({"Totals on disk": totals, "Rows by table on disk": counts, "Disk against log": vs_log,
                "Disk against baseline": vs_baseline},
               reports_dir / f"Assembly_{pd.Timestamp.today().date()}.md", title="Assembly")

# Assemble eligible tables

Builds the working datasets for the three London PICU sites (RBH, SMH, GOSH) from the raw supplier files.

**Input:** the raw site folders (read only) and the stay manifests in `derived/` (made in `manifests.ipynb`).

**What it does, for every table of a site:**
1. Keeps only rows that belong to a manifest stay (admitted 2013 or later, in PICANet, with real vitals).
2. Drops exact duplicate rows.
3. Writes one parquet file per table to `derived/assembled/<SITE>/`.

**What it does not do:** it drops no columns and changes no values. Every column is written as text, exactly as in the source. Anonymisation (new keys, relative times, free text, small cells) is a separate later step.

**Output:** row-level data. It stays in `derived/`, which is git-ignored, and never leaves the SDE. The logs printed here are counts only.

Run order: 1. `inventory_*` notebooks, 2. `manifests.ipynb`, 3. this notebook.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import inventory as I
import manifest as M
import assemble as A

smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
gosh_dir = Path("/home/jormond/Documents/DS217/users/jormond/GOSH")

In [ ]:
assembled_dir = open_picu_dir / "derived" /"assembled"
counts = A.parquet_counts(assembled_dir)
counts.groupby("site").agg(tables=("table", "count"), rows=("rows", "sum"))

In [ ]:
reports_dir = open_picu_dir / "derived" / "reports"
counts = A.parquet_counts(assembled_dir)
counts.to_csv(reports_dir / "Assembled_counts_baseline.csv", index=False)
counts

## Directories and manifests

Loads the three saved manifests. They hold one row per eligible stay and define what is kept:

- **RBH:** matched by `stay_key` (patient + admission time, with the NIC/NND prefix removed).
- **SMH:** matched by `encounterId`.
- **GOSH:** matched by `project_id` and time. A row belongs to the stay whose window (admission start to discharge) contains its timestamp. The supplier's `admission_identifier` is not used, because it was built from the archived PICANet file, whose times disagree with the main file's.

Everything is read as text so IDs are never altered (for example by dropping leading zeros).

In [ ]:
derived = open_picu_dir / "derived"
out_dir = derived / "assembled"
out_dir.mkdir(exist_ok=True)

rbh_m = pd.read_csv(derived / "RBH_manifest.csv", dtype=str)
smh_m = pd.read_csv(derived / "SMH_manifest.csv", dtype=str)
gosh_m = pd.read_csv(derived / "GOSH_manifest.csv", parse_dates=["adm_time", "disch_time"])

## Preview

Lists which tables will be assembled for each site (headers only, so it is quick), and which cannot be linked to a stay and will be skipped.

Defaults: RBH skips the old PICANet file; SMH skips the two medication files that `medications_v2` already combines; GOSH uses only the merged per-variable files (not raw `lab_components`, the archived PICANet file, the old parquet or the logs). The GOSH PICANet file (`eprweights`) is handled separately.

Check the list before the long runs below.

In [ ]:
for site, d in [("RBH", rbh_dir), ("SMH", smh_dir), ("GOSH", gosh_dir)]:
    print(site)
    A.list_tables(site, d)

## RBH

For each table: reads in chunks, converts `studyID` to the stay key, keeps rows of manifest stays, then removes duplicates. `studyID` is replaced by `stay_key`.

Two counts are reported separately in the log:
- **exact duplicates removed:** identical rows.
- **merged (prefix only):** rows that differ only in the NIC/NND prefix. The same stay can appear under both prefixes in different tables (about 2,500 stays in the sedation table), so these are copies of one record, not new information.

In [ ]:
rbh_log = A.assemble_site("RBH", rbh_dir, out_dir, rbh_m)
rbh_log

In [ ]:
rbh_summary = A.summarise_log(rbh_log)

## SMH

Keeps rows whose `encounterId` (`encounterID` in the medication files) is in the manifest, then removes exact duplicate rows.

Expect a lot of duplicates in the long-format tables (vent, vitals, airway, blood gas, labs): the inventory found 29 to 52% exact repeats there. Whether they are true repeats or distinct measurements with identical fields is still an open question, so this only removes rows that are identical in every column.

The vent and vitals tables have about 10M rows, so this is the slow one.

In [ ]:
smh_log = A.assemble_site("SMH", smh_dir, out_dir, smh_m)
smh_log

In [ ]:
smh_summary = A.summarise_log(smh_log)

## GOSH

GOSH has no stay key in its tables, so rows are attached to stays by patient and time:
- the patient comes from `project_id`;
- the stay is the one for that patient whose window (admission start, with no margin, to discharge) contains `observation_datetime`;
- rows outside every stay (other wards, theatres, readmission gaps) are dropped, and `stay_id` is added to the rest.

The `eprweights` PICANet table is written with all 28 columns plus `stay_id`.

Tables without `project_id` or `observation_datetime` are not assembled; they are listed at the end of the run for a decision.

This is the longest run, with tables of millions of rows. Reruns skip tables that already have an output file.

## Check the time columns of the balance and flowsheet tables

`Net_Bal`, `Net_Bal_24` and the ventilation-mode flowsheet rows each have `chartTime`, `start_datetime` and `end_datetime`. If the start and end columns are an admission window repeated on every row (as in the archived PICANet file), they must not be used to place rows in time.

This samples each table (the first 500,000 rows) and prints counts only: how many rows parse, the date range, how many patients have a single value in the column, and how often each pair of columns agree to the minute. "Patients with one distinct value" near 100% would mean an admission-level constant, not a row time.

In [ ]:
pick = r"caboodle_fluid_balance|Total_fluids"
A.list_tables("GOSH", gosh_dir, include=pick)
A.time_candidates("GOSH", gosh_dir, include=pick)

In [ ]:
for rel in ["merged_balance/Net_Bal.csv",
            "merged_balance/Net_Bal_24.csv"]:
    A.time_column_check(gosh_dir, rel, ["chartTime", "start_datetime", "end_datetime"])

In [ ]:
checks = {
    "merged_vent_data/vent_Ventilation_mode_flowsheet_rows.csv": ["chart_time", "start_datetime", "end_datetime"],
}
for rel, cols in checks.items():
    A.time_column_check(gosh_dir, rel, cols)

In [ ]:
time_cols = {
    "July2026extraction_by_alex/pAIC/merged_antibiotics/antibiotics.csv": "chartTime",
    "merged_balance/Net_Bal.csv": "chartTime",
    "merged_balance/Net_Bal_24.csv": "chartTime",
    "merged_balance/Total_fluids.csv": "chartTime",
    "merged_vent_data/vent_Ventilation_mode_flowsheet_rows.csv": "chart_time",
}
gosh_log = A.assemble_site("GOSH", gosh_dir, out_dir, gosh_m, time_cols=time_cols)
gosh_log

## Summarise the assembly logs

Prints totals for a site's log (rows read, rows in manifest stays, duplicates removed, rows written), lists the tables where duplicates were removed, and shows the tables that keep the smallest share of their rows. Counts only, so the output can be pasted and saved.

In [ ]:
gosh_summary = A.summarise_log(gosh_log)

In [ ]:
pat = "Endotracheal|Hb|Haemoglobin|neutrophils|Neutrophils|wbc|WBC"
gosh_log[gosh_log["table"].str.contains(pat)][["table", "rows read", "in manifest stays", "rows written"]]

## Save the logs

The logs hold counts only (rows read, rows in stays, duplicates, rows written), so they can go in `reports/`. They record how many rows each table lost at each step.

In [ ]:
reports = open_picu_dir / "reports"
reports.mkdir(exist_ok=True)
for name, log in [("RBH", rbh_log), ("SMH", smh_log), ("GOSH", gosh_log)]:
    log.to_csv(reports / f"{name}_assembly_log.csv", index=False)

## One row per stay? (demographics, outcomes, PICANet)

These tables should have one row per stay, but a stay can have several rows that are not exact duplicates (for example repeated demographics rows, or conflicting outcomes). This counts rows per stay for each table, and lists which columns take different values between the rows of the same stay. Counts only.

If the rows of a stay disagree on outcome or discharge status, we need a rule for choosing before anonymisation.

In [ ]:
for site, key, names in [
    ("RBH", "stay_key", ["ai_picu_demographics", "ai_picu_outcomes", "ai_picu_piccanet"]),
    ("SMH", "encounterId", ["ai_picu_demographics", "ai_picu_outcomes", "ai_picu_piccanet"]),
]:
    for n in names:
        A.per_stay_conflicts(out_dir / site / f"{n}.parquet", key)
        print()

## Do the demographics times agree with the PICANet times? (RBH and SMH)

At GOSH a stay is defined by the PICANet admission and discharge times. At RBH and SMH the manifest takes the admission time from the demographics table, and PICANet is used only to decide which stays are included. Before pooling the sites we need one stay definition, so this measures how far the two sources disagree.

For each site it prints counts only:
- **Admission time:** demographics minus PICANet, for every stay in both tables and for the manifest stays. It counts how many agree to the minute, differ by under an hour, by 1 to 24 hours, or by a day or more. It also counts differences of exactly one hour, which would point to a clock or time-zone mismatch.
- **Year effects:** how many manifest stays would fall before 2013 if PICANet times were used, and how many stays the two sources put in different years.
- **Discharge and length of stay:** whether the PICANet discharge time minus the admission time matches the PICANet length-of-stay column.

If a column name is wrong, it lists the date-like columns of the table and stops, so we can correct it.

In [ ]:
import picanet_times as PT

PT.check_times(rbh_dir, "RBH", demo_adm="PICUadm_dttm", los_col="LOSHours",
               los_unit="hours", manifest=rbh_m)
PT.check_times(smh_dir, "SMH", demo_adm="inTime", manifest=smh_m)

## RBH: which admission time and which discharge time?

Two things are unclear for RBH. The PICANet discharge time (`UnitDisDateTime`) comes before the admission time in 52% of stays and doesn't match the length-of-stay column, so either it isn't a discharge time or it is being parsed wrongly. And the demographics and PICANet admission times differ for 29% of stays.

**Discharge diagnostic.** Shows the format of the admission and discharge columns (letters as `a`, digits as `d`), then how discharge minus admission compares with `LOSHours`. A format mismatch, a day and month swap, or a column that isn't a discharge time would each leave a different signature. Counts only.

**First-vitals offset.** For each manifest stay, the hours from each candidate admission time (demographics, PICANet) to the first vitals row. The better t0 puts the first vitals closer to zero, so this is evidence for which time to use as the zero point of relative time.

In [ ]:
PT.discharge_diagnostic(rbh_dir, "RBH")
PT.first_vitals_offset(rbh_dir, "PICUadm_dttm", rbh_m)

## Length of stay at RBH and SMH

The PICANet discharge time is unusable at RBH, and SMH's PICANet table has no discharge column, so length of stay has to come from the length-of-stay columns. Two checks:

1. **Time-of-day test (RBH).** If the discharge column carries the right time of day but the wrong date, its time of day should match the time of day of admission plus `LOSHours`, and its date should equal the admission date. This counts how often each holds.
2. **Outcomes against PICANet.** Each site's outcomes table has its own length of stay (`picuLOS` at RBH, `LOS` at SMH). This compares it per stay with PICANet's `LOSHours`, in the same unit and as days against hours, and counts the stays where PICANet's value is missing but the outcomes value is present.

In [ ]:
PT.discharge_timeofday(rbh_dir, "RBH")
PT.check_los(rbh_dir, "RBH", out_los="picuLOS")
PT.check_los(smh_dir, "SMH", out_los="LOS")

## Stay-level tables: columns available at each site

Before writing the stay-level table (one row per stay, one schema for all three sites), this lists what each site's static tables actually contain: for every column, the non-null share, the number of distinct values, and either the shape of the values (letters as `a`, digits as `d`) or, for columns with 8 or fewer distinct values, the category counts. Counts under 10 are shown as `<10`.

The tables are the assembled demographics, outcomes and PICANet tables for RBH and SMH, and the `eprweights` PICANet table for GOSH. The output feeds the column mapping between sites, so I don't guess names.

In [ ]:
import static as S

for p in [
    out_dir / "RBH" / "ai_picu_demographics.parquet",
    out_dir / "RBH" / "ai_picu_outcomes.parquet",
    out_dir / "RBH" / "ai_picu_piccanet.parquet",
    out_dir / "SMH" / "ai_picu_demographics.parquet",
    out_dir / "SMH" / "ai_picu_outcomes.parquet",
    out_dir / "SMH" / "ai_picu_piccanet.parquet",
    out_dir / "GOSH" / "PICANet_eprweights.parquet",
]:
    S.describe_columns(p)

## Stay-level tables

One row per stay, one schema across the three sites, built from the assembled demographics, outcomes and PICANet tables by the agreed rules:

- PICANet fields first; demographics only for what PICANet lacks.
- Weight: PICANet weight if valid (0.3 to 200 kg; the 999 placeholder becomes missing), else the median of the stay's valid demographics weights. GOSH: `weight_picanet`, then `weight_epr`, then `AdmWeight`. `weight_source` records the source.
- Length of stay in hours: PICANet where present, else the outcomes table. `los_source` records the source.
- `t0` is the hospital's recorded PICU admission (RBH demographics, SMH `inTime`/PICANet `AdDateTime`, GOSH PICANet `AdDateTime`).
- `died` comes from the outcomes tables (RBH, SMH) or the PICANet discharge status (GOSH).
- Stays with two disagreeing PICANet rows are excluded (counts printed).
- Ethnicity is PICANet only; demographics vocabularies are kept as `x_` columns because they differ between sites.
- Free text is not carried into this table.
- Site-specific fields keep an `x_` prefix and are not part of the pooled schema.

The coverage table below shows, for every pooled column, how complete it is at each site. That is the first answer to "what survives pooling". The output is counts only.

In [ ]:
static = S.build_all(out_dir)
S.coverage(static)

## Save the stay-level tables

The tables hold original keys and exact admission times, so they are row-level data: `derived/static/`, git-ignored, inside the SDE. `pooled` stacks the common columns of the three sites for the pooling analysis.

In [ ]:
static_dir = derived / "static"
static_dir.mkdir(exist_ok=True)
for site, df in static.items():
    df.to_parquet(static_dir / f"{site}_static.parquet", index=False)
pooled = S.pool(static)
pooled.to_parquet(static_dir / "pooled_static.parquet", index=False)
pooled.groupby("site").size()

## Do the categorical vocabularies match across sites?

Ethnicity has 17, 19 and 19 categories at RBH, SMH and GOSH, and diagnosis group 13, 14 and 14. Before pooling these columns we need to see whether the labels are the same. This prints, for each column, the category counts of the three sites side by side. Counts under 10 are shown as `<10` and absent categories as `-`.

A category that exists at one site only is a re-identification risk in a pooled table, and a mismatch in spelling would split one category into two.

In [ ]:
for col in ["ethnicity", "diagnosis_group", "sex"]:
    print(col)
    S.category_table(static, col)
    print()

In [ ]:
rb = static["RBH"]
ns = rb["ethnicity"].isin(["Not stated", "Unknown"]) | rb["ethnicity"].isna()
S.category_table({"RBH, PICANet not stated": rb[ns]}, "x_ethnicity_demographics")

# PIM at GOSH

In [ ]:
S.pim_by_year(static)

### SMH measurement mappings
`smh_mappings.py` records which `interventionId` (and, for vitals, `interventionId_attributeId`) maps to which concept, taken from the earlier analysis notebooks. This step lists the mapping as a table and counts, for each concept, how many rows and stays it covers in the assembled SMH tables. It also lists IDs that are not mapped, so nothing is silently ignored. Output is counts and measurement codes only.

In [ ]:
import smh_mappings as SM
SM.to_table().groupby("table").size()

In [ ]:
cov = SM.check_coverage(out_dir)

### SMH ventilation and airway coverage
Same check as for the other SMH tables, now with the ventilation mapping and the airway type (interventionId 8039). It reports rows and stays per concept and any unmapped measurement codes. Counts only.

In [ ]:
cov = SM.check_coverage(out_dir, tables=("vent", "airway"))

### SMH drugs: coverage
Same check for the three drug tables. Medications are keyed by materialId; infusions by interventionId using a candidate map from the SQL (not confirmed). Counts only.

In [ ]:
import importlib; importlib.reload(SM)
cov = SM.check_coverage(out_dir, tables=("vitals", "vent", "airway", "medications", "infusions"))

### Identify the unmapped SMH IDs
For IDs without a mapping yet, this shows how many rows and stays they have, and either the value distribution (numeric) or the most common drug names (text). Text values seen fewer than 10 times are hidden. No patient data is shown.

In [ ]:
importlib.reload(SM)
SM.describe_ids(out_dir, "infusions", ["11858", "13245", "11999", "10706"], "Drug_terse")
SM.describe_ids(out_dir, "vitals", ["3406_57907"], "terseForm")

In [ ]:
SM.describe_ids(out_dir, "vitals",
                ["8010_59129", "8010_59130", "8073_64034", "30546_32366", "8070_26358",
                 "3406_57902", "2934_55833"], "terseForm")

In [ ]:
st = pd.read_parquet(f"{derived}/static/pooled_static.parquet")
(st.assign(le6=st.los_hours <= 6).groupby("site")["le6"].agg(["sum", "mean", "count"]))